# HGAT Matched Ablation — Results Analysis

Run `scripts/run_ablation.py` first to populate `runs/`. Then run this notebook.

CRyPTIC phenotypes: `data/cryptic/`; VCFs: `data/cryptic/vcf/` (see `data/cryptic/SOURCE.txt`).

Sections:
1. Load metrics table
2. AUROC/AUPRC bar plots per arm × drug
3. Pathway attention heatmap (arm D)
4. SNP → gene → pathway attribution for canonical mutations
5. Mechanism hit-rate comparison across arms

In [ ]:
import sys, json, copy
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import torch

# Allow running from the analysis/ subdirectory
ROOT = Path('.').resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RUNS_DIR = ROOT / 'runs'
METRICS_CSV = ROOT / 'metrics_table.csv'

ARM_LABELS = {
    'A': 'SNP-only',
    'B': 'Pathway-minimal',
    'C': 'Pathway+KO',
    'D': 'Pathway+KO+Drug',
}
ARM_COLORS = {'A': '#4878CF', 'B': '#6ACC65', 'C': '#D65F5F', 'D': '#B47CC7'}

DRUG_SHORT = {
    'INH_BINARY_PHENOTYPE': 'INH',
    'RIF_BINARY_PHENOTYPE': 'RIF',
    'EMB_BINARY_PHENOTYPE': 'EMB',
    'LEV_BINARY_PHENOTYPE': 'LEV',
    'MXF_BINARY_PHENOTYPE': 'MXF',
    'ETH_BINARY_PHENOTYPE': 'ETH',
    'KAN_BINARY_PHENOTYPE': 'KAN',
    'AMI_BINARY_PHENOTYPE': 'AMK',
    'CFZ_BINARY_PHENOTYPE': 'CFZ',
    'LZD_BINARY_PHENOTYPE': 'LZD',
    'BDQ_BINARY_PHENOTYPE': 'BDQ',
    'DLM_BINARY_PHENOTYPE': 'DLM',
}
PRIMARY_DRUGS = ['INH_BINARY_PHENOTYPE','RIF_BINARY_PHENOTYPE','EMB_BINARY_PHENOTYPE',
                 'LEV_BINARY_PHENOTYPE','MXF_BINARY_PHENOTYPE','ETH_BINARY_PHENOTYPE',
                 'KAN_BINARY_PHENOTYPE','AMI_BINARY_PHENOTYPE']

print('Setup done. ROOT:', ROOT)

## 1. Load metrics table

In [ ]:
if not METRICS_CSV.exists():
    # Generate from runs/ if not yet created
    import subprocess
    subprocess.run(['python', str(ROOT/'scripts'/'aggregate_results.py'),
                    '--runs-dir', str(RUNS_DIR), '--output', str(METRICS_CSV)],
                   check=True)

df = pd.read_csv(METRICS_CSV)
df['drug_short'] = df['drug'].map(DRUG_SHORT).fillna(df['drug'])
df['arm_label'] = df['arm'].map(ARM_LABELS).fillna(df['arm'])
print(f'{len(df)} rows, arms: {sorted(df.arm.unique())}, drugs: {sorted(df.drug_short.unique())}')
df.head()

## 2. AUROC & AUPRC bar plots (primary drugs)

In [ ]:
def plot_metric_by_arm_drug(df, metric='AUROC', drugs=None, title=None, figsize=(14,5)):
    sub = df[df['metric'] == metric].copy()
    if drugs:
        sub = sub[sub['drug'].isin(drugs)]
    arms = sorted(sub['arm'].unique())
    drug_list = [d for d in (drugs or PRIMARY_DRUGS) if d in sub['drug'].values]
    drug_shorts = [DRUG_SHORT.get(d, d) for d in drug_list]

    x = np.arange(len(drug_list))
    width = 0.8 / max(len(arms), 1)
    fig, ax = plt.subplots(figsize=figsize)

    for i, arm in enumerate(arms):
        arm_df = sub[sub['arm'] == arm].set_index('drug')
        means = [arm_df.loc[d, 'mean'] if d in arm_df.index else float('nan') for d in drug_list]
        stds  = [arm_df.loc[d, 'std']  if d in arm_df.index else 0.0 for d in drug_list]
        offset = (i - len(arms)/2 + 0.5) * width
        bars = ax.bar(x + offset, means, width*0.9, label=ARM_LABELS.get(arm, arm),
                      color=ARM_COLORS.get(arm, '#888'), alpha=0.85,
                      yerr=stds, capsize=3, error_kw={'linewidth': 1})

    ax.set_xticks(x)
    ax.set_xticklabels(drug_shorts, fontsize=11)
    ax.set_ylabel(metric, fontsize=12)
    ax.set_title(title or f'{metric} per drug across ablation arms', fontsize=13)
    ax.legend(fontsize=9, loc='lower right')
    ax.set_ylim(max(0, sub['mean'].min() - 0.05), min(1.02, sub['mean'].max() + 0.05))
    ax.axhline(1.0, color='gray', linestyle='--', linewidth=0.7)
    plt.tight_layout()
    fig.savefig(ROOT / f'analysis/{metric.lower()}_by_arm_drug.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_metric_by_arm_drug(df, metric='AUROC', drugs=PRIMARY_DRUGS)
plot_metric_by_arm_drug(df, metric='AUPRC', drugs=PRIMARY_DRUGS)

In [ ]:
# Compact AUROC table
auroc_df = df[df['metric']=='AUROC'].copy()
auroc_df['val'] = auroc_df.apply(lambda r: f"{r['mean']:.4f}±{r['std']:.4f}", axis=1)
pivot = auroc_df.pivot(index='drug_short', columns='arm', values='val')
pivot = pivot.reindex(columns=sorted(pivot.columns))
# Order rows by primary drugs first
primary_shorts = [DRUG_SHORT[d] for d in PRIMARY_DRUGS if d in DRUG_SHORT]
ordered = [d for d in primary_shorts if d in pivot.index] + \
          [d for d in pivot.index if d not in primary_shorts]
pivot = pivot.reindex(ordered)
print("AUROC (mean±std) — primary drugs marked with *")
pivot

## 3. Pathway attention heatmap (arm D)

In [ ]:
# Reload arm D model and data to extract pathway attentions
# This cell requires the training run to have completed for arm D.

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
KEGG_JSON = str(ROOT / 'kegg_data/tb_knowledge_graph_full.json')

from amr_hgat.data import load_data, DRUG_COLS_PRIMARY
from amr_hgat.graph_builders import build_graph
from amr_hgat.model import build_model
import json

print('Loading data for interpretability analysis...')
data_bundle = load_data(
    vcf_dir=str(ROOT / 'data' / 'cryptic' / 'vcf'),
    phenotype_csv=str(ROOT / 'data' / 'cryptic' / 'CRyPTIC_reuse_table_20231208.csv'),
    drug_cols=DRUG_COLS_PRIMARY,
)
drug_cols = data_bundle['drug_cols']

data_D = build_graph(
    arm='D',
    isolate_snp_df=data_bundle['isolate_snp_df'],
    phenotype_df=data_bundle['phenotype_df'],
    snp_embeddings=data_bundle['snp_embeddings'],
    snp_list=data_bundle['snp_list'],
    drug_cols=drug_cols,
    kegg_json_path=KEGG_JSON,
)
print('Arm D graph built.')
print('Pathway count:', len(data_D['pathway'].pathway_ids) if hasattr(data_D['pathway'], 'pathway_ids') else 'N/A')

In [ ]:
model_D = build_model('D', data_D, drug_cols=drug_cols, hidden_channels=128)

# Try loading best checkpoint from fold 0
ckpt_path = RUNS_DIR / 'D' / 'fold_0.json'
if ckpt_path.exists():
    print(f'fold_0 metrics:', json.load(open(ckpt_path)))
else:
    print('[warn] No checkpoint found; model has random weights. Run run_ablation.py first.')

model_D = model_D.to(DEVICE)
data_D = data_D.to(DEVICE)

@torch.no_grad()
def get_pathway_attention_matrix(model, data, drug_cols):
    """Return [n_drugs, n_pathways] mean attention matrix."""
    model.eval()
    h1 = model.layer1(data.x_dict, data.edge_index_dict)
    h2 = model.layer2(h1, data.edge_index_dict, exclude_edge_types=model.layer2_exclude or None)
    iso_l2 = h2.get('isolate')
    p = h2.get('pathway')
    if iso_l2 is None or p is None:
        return None, None

    attn_matrix = np.zeros((len(drug_cols), p.size(0)))
    for d_i, drug_col in enumerate(drug_cols):
        short = drug_col.replace('_BINARY_PHENOTYPE','').replace('-','_')
        buf_name = 'mask_' + short
        mask = getattr(model, buf_name, torch.ones(p.size(0), device=p.device))
        attn_head = model.drug_attn_heads.get(short) if hasattr(model,'drug_attn_heads') else None
        if attn_head is not None:
            scores = attn_head(iso_l2.unsqueeze(1).expand(-1,p.size(0),-1)).squeeze(-1)  # [n_iso, n_pwy]
        else:
            scores = torch.matmul(iso_l2, p.T) / (p.shape[1]**0.5)
        scores = scores.masked_fill(mask.unsqueeze(0).expand_as(scores)==0, float('-inf'))
        alpha = torch.softmax(scores, dim=1)  # [n_iso, n_pwy]
        attn_matrix[d_i] = alpha.mean(0).cpu().numpy()

    pathway_ids = list(data['pathway'].pathway_ids)
    return attn_matrix, pathway_ids

attn_mat, pathway_ids = get_pathway_attention_matrix(model_D, data_D, drug_cols)
if attn_mat is not None:
    print(f'Attention matrix shape: {attn_mat.shape}  ({len(pathway_ids)} pathways)')

In [ ]:
if attn_mat is not None:
    from amr_hgat.data import DRUG_SHORT
    DRUG_SHORT_LOCAL = {
        'INH_BINARY_PHENOTYPE':'INH','RIF_BINARY_PHENOTYPE':'RIF',
        'EMB_BINARY_PHENOTYPE':'EMB','LEV_BINARY_PHENOTYPE':'LEV',
        'MXF_BINARY_PHENOTYPE':'MXF','ETH_BINARY_PHENOTYPE':'ETH',
        'KAN_BINARY_PHENOTYPE':'KAN','AMI_BINARY_PHENOTYPE':'AMK',
    }

    # Focus on top 20 pathways by max attention across drugs
    top_k = 20
    max_attn = attn_mat.max(axis=0)
    top_pwy_idx = np.argsort(max_attn)[-top_k:][::-1]
    top_pwy = [pathway_ids[i] for i in top_pwy_idx]
    sub_mat = attn_mat[:, top_pwy_idx]

    drug_labels = [DRUG_SHORT_LOCAL.get(d, d.replace('_BINARY_PHENOTYPE','')) for d in drug_cols]

    fig, ax = plt.subplots(figsize=(max(12, len(top_pwy)*0.6), max(5, len(drug_cols)*0.6)))
    im = ax.imshow(sub_mat, aspect='auto', cmap='YlOrRd', interpolation='nearest')
    ax.set_xticks(range(len(top_pwy)))
    ax.set_xticklabels(top_pwy, rotation=55, ha='right', fontsize=8)
    ax.set_yticks(range(len(drug_labels)))
    ax.set_yticklabels(drug_labels, fontsize=10)
    ax.set_title('Mean pathway attention per drug (Arm D)', fontsize=13)
    plt.colorbar(im, ax=ax, shrink=0.7, label='Mean attention')
    plt.tight_layout()
    fig.savefig(ROOT / 'analysis/pathway_attention_heatmap_D.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Expected dominant pathways: RIF->mtu03020, INH->mtu00074, EMB->mtu00572, AMK/KAN->mtu03010')

## 4. SNP → gene → pathway attribution for canonical resistance mutations

In [ ]:
# Known canonical mutations and their expected pathway associations
CANONICAL = [
    {'snp_prefix': 'katG_G2154',   'drug': 'INH', 'expected_pathway': 'mtu00074', 'note': 'katG S315T'},
    {'snp_prefix': 'rpoB_C761',    'drug': 'RIF', 'expected_pathway': 'mtu03020', 'note': 'rpoB S450L'},
    {'snp_prefix': 'embB_A917',    'drug': 'EMB', 'expected_pathway': 'mtu00572', 'note': 'embB M306V'},
    {'snp_prefix': 'gyrA_T280',    'drug': 'LEV', 'expected_pathway': None,       'note': 'gyrA D94G'},
    {'snp_prefix': 'rrs_A4326',    'drug': 'KAN', 'expected_pathway': 'mtu03010', 'note': 'rrs A1401G'},
]

@torch.no_grad()
def snp_attention_for_gene(model, data, gene_type, device=DEVICE):
    """Return {snp_id: mean attention} for a given gene type across all layers."""
    model.eval()
    result = {}
    for layer_i, layer in enumerate([model.layer1, model.layer2]):
        for (src, rel, dst), conv in layer.conv.convs.items():
            if src != 'isolate' or dst != gene_type:
                continue
            ei = data.edge_index_dict.get((src, rel, dst))
            if ei is None: continue
            x_src = data.x_dict.get(src)
            x_dst = data.x_dict.get(dst)
            if x_src is None or x_dst is None: continue
            try:
                _, (_, attn) = conv((x_src, x_dst), ei, return_attention_weights=True)
            except Exception:
                continue
            attn_mean = attn.mean(dim=1).cpu().numpy()
            snp_ids = getattr(data[dst], 'snp_ids', [])
            dst_nodes = ei[1].cpu().numpy()
            n_snp = data[dst].x.shape[0]
            agg = np.zeros(n_snp); cnt = np.zeros(n_snp)
            for e_i, si in enumerate(dst_nodes):
                agg[si] += attn_mean[e_i]; cnt[si] += 1
            avg = np.divide(agg, cnt, where=cnt>0, out=np.zeros_like(agg))
            for j, s in enumerate(snp_ids):
                result[s] = result.get(s, 0.0) + float(avg[j])
    return result

print('Attribution functions defined.')

In [ ]:
import json as _json
kg = _json.load(open(KEGG_JSON))
gene_to_pathways = kg.get('gene_to_pathways', {})

print('=== SNP attribution for canonical mutations ===\n')
for case in CANONICAL:
    gene = case['snp_prefix'].split('_')[0]
    print(f"--- {case['note']} ({case['drug']}) ---")
    attn = snp_attention_for_gene(model_D, data_D, gene)
    if not attn:
        print('  No attention data (gene not in graph or no edges).')
        continue
    top = sorted(attn.items(), key=lambda x: -x[1])[:10]
    for s, a in top:
        marker = ' <<<' if case['snp_prefix'] in s else ''
        print(f'  {s:<30} {a:.5f}{marker}')

    # Pathway projection
    gene_paths = gene_to_pathways.get(gene, [])
    if gene_paths:
        print(f'  Expected pathways via KEGG: {gene_paths}')
    exp_pwy = case['expected_pathway']
    print(f'  Expected dominant pathway: {exp_pwy}')
    print()

## 5. Mechanism hit-rate comparison across arms

In [ ]:
# Load all arm summaries and compute mean AUROC delta vs arm A
all_arms_path = RUNS_DIR / 'all_arms_summary.json'
if not all_arms_path.exists():
    print('[warn] all_arms_summary.json not found. Run run_ablation.py first.')
else:
    all_summ = json.load(open(all_arms_path))
    rows = []
    for arm, summ in all_summ.items():
        for drug, metrics in summ.items():
            if drug.startswith('_'): continue
            auroc_mean = metrics.get('AUROC', {}).get('mean', float('nan'))
            auroc_std  = metrics.get('AUROC', {}).get('std', float('nan'))
            rows.append({'arm': arm, 'drug': DRUG_SHORT.get(drug,drug), 'AUROC_mean': auroc_mean, 'AUROC_std': auroc_std})
    cmp_df = pd.DataFrame(rows)

    # Delta vs arm A
    base = cmp_df[cmp_df['arm']=='A'][['drug','AUROC_mean']].rename(columns={'AUROC_mean':'base_auroc'})
    cmp_df = cmp_df.merge(base, on='drug', how='left')
    cmp_df['delta'] = cmp_df['AUROC_mean'] - cmp_df['base_auroc']
    cmp_df['sig'] = cmp_df['delta'].abs() > cmp_df['AUROC_std']

    # Heatmap of deltas
    pivot_delta = cmp_df[cmp_df['arm'] != 'A'].pivot(index='drug', columns='arm', values='delta')
    fig, ax = plt.subplots(figsize=(8, max(4, len(pivot_delta)*0.5)))
    vmax = max(0.05, pivot_delta.abs().max().max())
    im = ax.imshow(pivot_delta.values, aspect='auto', cmap='RdYlGn',
                   vmin=-vmax, vmax=vmax, interpolation='nearest')
    ax.set_xticks(range(len(pivot_delta.columns)))
    ax.set_xticklabels([ARM_LABELS.get(c,c) for c in pivot_delta.columns], fontsize=10)
    ax.set_yticks(range(len(pivot_delta.index)))
    ax.set_yticklabels(pivot_delta.index, fontsize=10)
    ax.set_title('AUROC delta vs arm A (green=better, red=worse)', fontsize=12)
    plt.colorbar(im, ax=ax, shrink=0.8, label='ΔAUROC')
    # Annotate cells
    for i in range(pivot_delta.shape[0]):
        for j in range(pivot_delta.shape[1]):
            v = pivot_delta.values[i,j]
            if not np.isnan(v):
                ax.text(j, i, f'{v:+.3f}', ha='center', va='center', fontsize=8,
                        color='black' if abs(v) < vmax*0.6 else 'white')
    plt.tight_layout()
    fig.savefig(ROOT / 'analysis/auroc_delta_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(cmp_df[['arm','drug','AUROC_mean','AUROC_std','delta']].to_string(index=False))

In [ ]:
# Known mechanism gene sets per drug
KNOWN_MECHANISMS = {
    'INH': {'katG','inhA','ahpC','fabG1','ndh'},
    'RIF': {'rpoB','rpoC'},
    'EMB': {'embA','embB','embC','iniA','iniB','iniC'},
    'PZA': {'pncA','rpsA'},
    'LEV': {'gyrA','gyrB'},
    'MXF': {'gyrA','gyrB'},
    'KAN': {'rrs','eis'},
    'AMK': {'rrs','eis'},
    'ETH': {'ethA','ethR','inhA'},
}

# Compute mechanism hit rate for arm D (and A if available)
def mechanism_hit_rate(model, data, drug_col, top_k=10):
    """Fraction of top_k attended genes per drug that are in known mechanism set."""
    short = {'INH_BINARY_PHENOTYPE':'INH','RIF_BINARY_PHENOTYPE':'RIF','EMB_BINARY_PHENOTYPE':'EMB',
             'LEV_BINARY_PHENOTYPE':'LEV','MXF_BINARY_PHENOTYPE':'MXF','ETH_BINARY_PHENOTYPE':'ETH',
             'KAN_BINARY_PHENOTYPE':'KAN','AMI_BINARY_PHENOTYPE':'AMK'}.get(drug_col, drug_col)
    known = KNOWN_MECHANISMS.get(short, set())
    if not known: return float('nan'), [], []

    gene_scores = {}
    snp_gene_types = [nt for nt in data.node_types
                      if nt not in ('isolate','pathway','gene','ko','drug')]
    for gene_type in snp_gene_types:
        attn = snp_attention_for_gene(model, data, gene_type)
        if attn:
            gene_scores[gene_type] = sum(attn.values())

    top_genes = sorted(gene_scores, key=gene_scores.get, reverse=True)[:top_k]
    overlap = sorted(set(top_genes) & known)
    return len(overlap) / max(1, len(known)), top_genes, overlap

print('=== Mechanism hit rates (Arm D) ===')
for dcol in drug_cols:
    rate, top_genes, overlap = mechanism_hit_rate(model_D, data_D, dcol)
    short = DRUG_SHORT_LOCAL.get(dcol, dcol)
    print(f'  {short:<4}  hit_rate={rate:.2f}  overlap={overlap}')